# Formula 1 regulation PDF Q&A

This notebook builds a small retrieval-augmented question-answering system, one step at a time. We will extract text from the PDFs in documents/, split it into chunks, embed the chunks, retrieve passages relevant to a question, and ask a chat model to answer from those passages with page citations.

The first embedding run sends the document text to the OpenAI API.

## 1. Load libraries and API key

LangChain's PyMuPDFLoader reads the PDFs into page-level Documents with source and page metadata. RecursiveCharacterTextSplitter then splits those Documents while preserving the metadata.


In [1]:
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()


/var/folders/xw/sr9jsbqd6yqfw_bfrwthchy00000gn/T/ipykernel_3449/3830965410.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader


True

## 2. Find the PDFs

Put the PDFs you want to search in documents/. We keep each source filename so answers can cite where a passage came from.

In [2]:
pdf_dir = Path("documents")
pdf_paths = sorted(pdf_dir.glob("*.pdf"))

if not pdf_paths:
    raise FileNotFoundError(f"No PDF files found in {pdf_dir.resolve()}")

print(f"Found {len(pdf_paths)} PDFs:")
for path in pdf_paths:
    print("-", path.name)


Found 6 PDFs:
- FIA 2026 F1 Regulations - Section A [General Provisions] - Iss 03 - 2026-06-25.pdf
- FIA 2026 F1 Regulations - Section B [Sporting] - Iss 09 - 2026-10-01.pdf
- FIA 2026 F1 Regulations - Section C [Technical] - Iss 21 - 2026-10-19.pdf
- FIA 2026 F1 Regulations - Section D [Financial - F1 Teams] - Iss 08 - 2026-10-01.pdf
- FIA 2026 F1 Regulations - Section E [Financial – PU Manufacturers] - Iss 07 - 2026-10-19.pdf
- FIA 2026 F1 Regulations - Section F [Operational] - Iss 11 - 2026-10-01.pdf


## 3. Load each PDF page with LangChain

DirectoryLoader finds every PDF in the folder, and PyMuPDFLoader extracts its text page by page. Each result is a LangChain Document: its page_content holds the text, and metadata keeps the source PDF and zero-based page index.


In [3]:
loader = DirectoryLoader(
    path=str(pdf_dir),
    glob="*.pdf",
    loader_cls=PyMuPDFLoader,
)
docs = loader.load()

if not docs:
    raise ValueError("The PDFs were found, but no pages were loaded.")

print(f"Loaded {len(docs)} pages from {len(pdf_paths)} PDFs.")
print(docs[0].metadata)
print(docs[0].page_content[:500])


Loaded 613 pages from 6 PDFs.
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-10-19T15:37:26+02:00', 'source': 'documents/FIA 2026 F1 Regulations - Section E [Financial – PU Manufacturers] - Iss 07 - 2026-10-19.pdf', 'file_path': 'documents/FIA 2026 F1 Regulations - Section E [Financial – PU Manufacturers] - Iss 07 - 2026-10-19.pdf', 'total_pages': 61, 'format': 'PDF 1.7', 'title': '', 'author': 'FIA', 'subject': '', 'keywords': '', 'moddate': '2026-10-19T15:37:26+02:00', 'trapped': '', 'modDate': "D:20261019153726+02'00'", 'creationDate': "D:20261019153726+02'00'", 'page': 0}
SECTION E: FINANCIAL REGULATIONS (POWER UNIT MANUFACTURERS) 
 
 
 
 
0  
E 
E1 
2026 Formula 1 Regulations – Section E [Financial - Power Unit Manufacturers] 
©2026 Fédération Internationale de l’Automobile 
19 October 2026
Issue 07
SECTION E: FINANCIAL REGULATIONS  
(POWER UNIT MANUFACTURERS) 
 
Version: 
 
 
Issue 07 
Status:  
 
 
PUBLISHE

## 4. Split pages into overlapping chunks with LangChain

RecursiveCharacterTextSplitter tries natural boundaries such as paragraphs and lines before falling back to smaller separators. split_documents() returns LangChain Documents and carries their source and page metadata into each chunk.


In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,  # characters
    chunk_overlap=200,  # characters shared by neighboring chunks
)

chunks = text_splitter.split_documents(docs)

if not chunks:
    raise ValueError("No text could be extracted. Scanned PDFs may need OCR.")

print(f"Created {len(chunks)} chunks.")
print(chunks[0].metadata)
print(chunks[0].page_content[:500])


Created 1813 chunks.
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-10-19T15:37:26+02:00', 'source': 'documents/FIA 2026 F1 Regulations - Section E [Financial – PU Manufacturers] - Iss 07 - 2026-10-19.pdf', 'file_path': 'documents/FIA 2026 F1 Regulations - Section E [Financial – PU Manufacturers] - Iss 07 - 2026-10-19.pdf', 'total_pages': 61, 'format': 'PDF 1.7', 'title': '', 'author': 'FIA', 'subject': '', 'keywords': '', 'moddate': '2026-10-19T15:37:26+02:00', 'trapped': '', 'modDate': "D:20261019153726+02'00'", 'creationDate': "D:20261019153726+02'00'", 'page': 0}
SECTION E: FINANCIAL REGULATIONS (POWER UNIT MANUFACTURERS) 
 
 
 
 
0  
E 
E1 
2026 Formula 1 Regulations – Section E [Financial - Power Unit Manufacturers] 
©2026 Fédération Internationale de l’Automobile 
19 October 2026
Issue 07
SECTION E: FINANCIAL REGULATIONS  
(POWER UNIT MANUFACTURERS) 
 
Version: 
 
 
Issue 07 
Status:  
 
 
PUBLISHED 
Date: 

## 5. Create an embedding for each chunk

An embedding is a numeric representation of text meaning. We use text-embedding-3-small here. This call can take a while and incurs API usage; rerunning this cell creates the embeddings again.

In [5]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
chunk_vectors = embeddings.embed_documents(
    [chunk.page_content for chunk in chunks]
)

print(f"Created {len(chunk_vectors)} vectors.")
print(f"Each vector has {len(chunk_vectors[0])} values.")


Created 1813 vectors.
Each vector has 1536 values.


## 6. Retrieve passages for a question

We embed the question, compare it with the document vectors using cosine similarity, and keep the top matches. These passages become the evidence for the answer.

In [6]:
query = "What limits apply to aerodynamic testing?"
query_vector = embeddings.embed_query(query)
scores = cosine_similarity([query_vector], chunk_vectors)[0]
top_k = 5
best_indices = sorted(range(len(scores)), key=scores.__getitem__, reverse=True)[:top_k]
retrieved = [(chunks[index], float(scores[index])) for index in best_indices]

for chunk, score in retrieved:
    source = Path(chunk.metadata["source"]).name
    page_number = int(chunk.metadata["page"]) + 1
    print(f"{score:.3f} | {source}, page {page_number}")
    print(chunk.page_content[:400], "\n")


0.673 | FIA 2026 F1 Regulations - Section F [Operational] - Iss 11 - 2026-10-01.pdf, page 15
conducted by an F1 car during and at track testing as defined by Article B11 of the Sporting 
Regulations will not be considered as Restricted Aerodynamic testing. 
F4.5.2 
Wind tunnel testing solely for the development of power unit heat exchangers that reject heat to air, 
or the running of the power unit from a boundary commencing at the power unit air intake ducts, 
passing through the power u 

0.672 | FIA 2026 F1 Regulations - Section F [Operational] - Iss 11 - 2026-10-01.pdf, page 15
shape or contact patch) and flap angles of the adjustable regions of the Front Wing Profiles (as 
defined by C3.10.10 of the Technical Regulations) or incidence of the RW Flap (as defined by 
C3.11.6 of the Technical Regulations) are permitted. Furthermore, when using an FIA approved 
geometry the FIA may approve additional incidence changes to bodywork (“active aerodynamics”) 
which will be clear 

0.633 | 

## 7. Generate a cited answer

The model receives only the retrieved passages. The prompt asks it to say when the evidence is missing and to cite claims with the PDF filename and page.

In [7]:
context = "\n\n---\n\n".join(
    (
        f"Source: {Path(chunk.metadata['source']).name}, "
        f"page {int(chunk.metadata['page']) + 1}\n"
        f"{chunk.page_content}"
    )
    for chunk, _score in retrieved
)

prompt = f"""Answer the question using only the source passages below.
If the passages do not contain the answer, say you could not find it in the provided documents.
Cite supporting statements with the source filename and page, for example [filename.pdf, p. 12].

Question: {query}

Source passages:
{context}"""

chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
response = chat_model.invoke(prompt)
print(response.content)


The limits that apply to aerodynamic testing are outlined in the FIA 2026 F1 Regulations. Here are the key points:

1. **Definition of Restricted Aerodynamic Testing**: Restricted Aerodynamic testing is defined as testing conducted by an F1 Team or its associates to measure, observe, or infer any aerodynamic forces, displacements, pressures, or airflow direction resulting from incident airflow [filename.pdf, p. 8].

2. **Conditions for Exemptions**: Certain tests are exempt from being classified as Restricted Aerodynamic testing:
   - Wind tunnel testing for power unit heat exchangers is exempt if there is no direct or indirect measurement of aerodynamic force [filename.pdf, p. 15].
   - Steady state and dynamic engine dynamometer work is exempt if the bodywork used does not include front or rear wing assemblies and no devices for measuring aerodynamic forces or flow characteristics are installed [filename.pdf, p. 15].

3. **Restrictions on Modifications**: A three-dimensional represen

## Try another question

Change query in step 6 and rerun steps 6 and 7. The PDFs and their embeddings remain available in the current notebook session.